# 情報数理 基礎Ⅰ演習 第 5 回｜細かく足して、たまった量を出す

スライドの札にあるセルを、上から順に実行します（セルを選んで `Shift` + `Enter`）。

- 🔹 のセルは、実行するだけです
- ✏️ のセルは、`____` を埋めてから実行します

コードは半角で打ちます。

## 0. 提出のしかた

1. Colab の左上で、ファイル名を suuri05_学籍番号.ipynb に変えます（学籍番号は半角で打ちます）。名前を変えられないときは、［ファイル］→［ドライブにコピーを保存］を選んでから変えます
2. ✏️ の 5 つのセルを埋めて、実行します
3. ［ファイル］→［ダウンロード］→［.ipynb をダウンロード］で保存します
4. Classroom の第 5 回の課題に添付して、［提出］を押します

## Python で積分を書く（🔹）

### 🔹 ノート P1 for で足していく（スライド 5-1）

2-1 の説明用の値から、最初の 6 本を使います。rain の各値は、棒 1 本ぶん（10 分間）の雨量です。棒の高さ（mm/h）に時間幅を掛けた量（mm）なので、もう一度掛けません。出力の左が今回の x、右が足したあとの total です。第 4 回の for の続きです。

In [ ]:
rain = [0.5, 0.5, 1, 1.5, 2, 3]
total = 0
for x in rain:
    total += x
    print(x, total)

**数を替えてみる**

`rain = [0.5, 0.5, 1]` に替えると、足す回数が 3 回になり、最後の出力は `1 2.0` です。

### 🔹 ノート P2 短冊の面積を出す関数（スライド 5-2）

関数 riemann は、関数 f と両端 a・b、本数 n を受け取って短冊の和を返します。`lambda x: x**3` は x³ を返す関数です。右端で取る場合は `x = a + (k + 1) * h`、真ん中で取る場合は `x = a + (k + 0.5) * h` に替えます。

In [ ]:
def riemann(f, a, b, n):
    h = (b - a) / n
    total = 0
    for k in range(n):
        x = a + k * h
        total += f(x) * h
    return total
n = 10
print(riemann(lambda x: x**3, 0, 2, n))

**数を替えてみる**

`n = 20` に替えると、出力は `3.610000000000001` です。幅を半分にした左端の和です。

### 🔹 ノート P3 台形で足すと、本数を倍にするたびに外れが 4 分の 1 になる（スライド 5-3）

台形公式では、両端の高さを半分ずつ足し、内側の高さを全部足してから幅を掛けます。出力は、本数・和・4 からの外れ（誤差）・前の外れとの比です。round で小数第 6 位まで丸めます。前期第 11 回の補足の台形則と、第 4 回の近似の続きです。

In [ ]:
def trapezoid(f, a, b, n):
    h = (b - a) / n
    total = (f(a) + f(b)) / 2
    for k in range(1, n):
        total += f(a + k * h)
    return total * h
previous = None
for n in [10, 20, 40, 80]:
    value = trapezoid(lambda x: x**3, 0, 2, n)
    error = abs(value - 4)
    ratio = error / previous if previous else None
    print(n, round(value, 6), round(error, 6),
          round(ratio, 6) if ratio else "-")
    previous = error

**数を替えてみる**

本数を `[12, 24, 48, 96]` に替えて実行すると、最後の行は `96 4.000434 0.000434 0.25` です。

### 🔹 ノート P4 SymPy で不定積分と定積分を出す（スライド 5-4）

SymPy の integrate は、区間を付けない場合は原始関数を 1 つ返します。`(x, 0, 3)` を付けると定積分です。`sp.diff(F, x)` は F を微分した式を返します。第 4 回の SymPy の続きです。

In [ ]:
import sympy as sp
x = sp.symbols("x")
F = sp.integrate(x**2, x)
print(F)
print(sp.integrate(x**2, (x, 0, 3)))
print(sp.diff(F, x))
print(sp.integrate(sp.exp(-x**2), x))

**数を替えてみる**

定積分の区間を `(x, 0, 2)` に替えると、2 行目の出力は `8/3` です。

### 🔹 ノート P5 原始関数が書けない e⁻ˣ² の積分を数で出す（スライド 5-5）

e⁻ˣ² の原始関数は初等関数では書けません。SymPy は erf を含む式を返します。quad は原始関数を使わずに積分を計算し、近似値と誤差の見積もりを組（タプル）で返します。2 番目は絶対誤差の推定値で、実際の誤差や保証された上限ではありません。

In [ ]:
import math
import sympy as sp
from scipy.integrate import quad
x = sp.symbols("x")
print(sp.integrate(sp.exp(-x**2), (x, 0, 1)))
def f(x):
    return math.exp(-x**2)
print(quad(f, 0, 1))

**数を替えてみる**

最後の行を `print(quad(f, 0, 2))` に替えると、出力は `(0.8820813907624216, 9.793070696178203e-15)` です。SymPy の行の区間は変わりません。

## スライドの続き（🔹）

### 🔹 ノート P6 微分して戻る関数を探す（原始関数）（スライド 6-2）

表の原始関数を integrate で求め、diff で微分します。最後の列が 0 なら、微分の結果が元の関数と一致します。このセルは x ＞ 0、n は正の整数として確かめます。

In [ ]:
import sympy as sp
x = sp.symbols("x", positive=True)
n = sp.symbols("n", positive=True, integer=True)
for f in [x**n, 1/x, sp.exp(x), sp.sin(x), sp.cos(x)]:
    F = sp.integrate(f, x)
    print(f, F, sp.simplify(sp.diff(F, x) - f))

**数を替えてみる**

リストの先頭を `[x**4,` に替えると、1 行目の出力は `x**4 x**5/5 0` です。

### 🔹 ノート P7 下にある面積は負に数える（スライド 6-4）

sin x の 0 から 2π までの定積分と、e²ˣ の原始関数を出します。角はラジアンです。

In [ ]:
import sympy as sp
x = sp.symbols("x")
print(sp.integrate(sp.sin(x), (x, 0, 2*sp.pi)))
print(sp.integrate(sp.exp(2*x), x))

**数を替えてみる**

`sp.exp(3*x)` に替えると、2 行目の出力は `exp(3*x)/3` です。

## 演習（✏️）

`____` を埋めて実行します。ヒントは、スライド 7-1・7-2 にあります。

### ✏️ ノート 7-1 問 1

$\int_{1}^{2}\left(3x^{2}+2x\right)dx$ を求めます。

In [ ]:
import sympy as sp
x = sp.symbols("x")
f = 3*x**2 + 2*x
result = ____
print("定積分:", result)

### ✏️ ノート 7-1 問 2

$\int_{0}^{1}e^{2x}\,dx$ を求めます。

In [ ]:
import sympy as sp
x = sp.symbols("x")
f = sp.exp(2*x)
G = ____
result = G.subs(x, 1) - G.subs(x, 0)
print("定積分:", result, float(result))

### ✏️ ノート 7-1 問 3

$\int_{0}^{4}\left(x-3\right)dx$ を求めます。

In [ ]:
import sympy as sp
x = sp.symbols("x")
f = x - 3
F = sp.integrate(f, x)
result = ____
print("定積分:", result)

### ✏️ ノート 7-2 問 1

電力が 2 kW から 0.55 kW へ 30 分で直線的に下がります。電力量（kWh）と、目安の単価 31 円/kWh での電力量料金を求めます。

In [ ]:
kw0 = 2
kw1 = 0.55
minutes = 30
unit_price = 31
energy = ____
print("電力量:", energy)
print("電力量料金:", energy * unit_price)

### ✏️ ノート 7-2 問 2

元金 1,200,000 円・年利 2.4%・24 回の元金均等返済で、利息の総額を求めます。年利を 12 で割った月利を、各月の返済前の残高に掛けます。手数料と途中の丸めは含めません。

In [ ]:
principal = 1200000
annual_rate = 0.024
months = 24
monthly_rate = annual_rate / 12
repayment = principal / months
interest = 0
for k in range(months):
    balance = principal - repayment * k
    interest += ____
print("利息の総額:", interest)